# 03 — Data Correction Checks

This notebook tests **whether all the data-correction transformations are conducted** and **whether the validation tests are working**.

It runs the corrections from `src/correct_data_issues.py` on the first-transaction churn table, then runs every validation check from `src/data_validation_tests.py` to confirm that each issue we decided to correct is fully resolved. If any correction is incomplete, `validate_all` raises a `DataValidationError` listing every check that did not pass.

Not every issue found in `02_data_quality_first_txn.ipynb` requires a transformation — three of them are deliberately left untouched, so there is no correction and no check for those. The table below states which is which.

## Which changes need to be made

All 8 issues from `02_data_quality_first_txn.ipynb`, in the same numbering as its Resolution plan. **5 need a transformation; 3 are left as they are on purpose.**

| # | Issue | Decision | Correction | Validation check |
|---|---|---|---|---|
| 1 | Missing `customer_id` / `description` | leave — already resolved | — | — |
| 2 | Zero prices (`price == 0`) | **correct** | `remove_zero_prices` | `validate_no_zero_prices` |
| 3 | One stock code → many descriptions | **correct** | `normalize_descriptions` | `validate_consistent_descriptions` |
| 4 | One description → many stock codes | leave — by design | — | — |
| 5 | Invalid stock-code formats | **correct** | `remove_invalid_stock_codes` | `validate_no_invalid_stock_codes` |
| 6 | Cancellations (`C` invoices) | **correct** | `remove_cancellations` | `validate_no_cancellations` |
| 7 | Outliers | leave — by design | — | — |
| 8 | Test products (`TEST*`) | **correct** | `remove_test_products` | `validate_no_test_products` |

### Why the three untouched issues need no transformation

**1 — Missing values.** Already resolved upstream, not by a decision to ignore it. `prepare_churn_data.py` drops every row with a missing `customer_id` when it builds the first-transaction table, and no missing `description` survives either: §1 of `02` measures both at 0.00%, down from 22.77% and 0.41% in the raw data. There is nothing left to correct.

**4 — One description → many stock codes.** `stock_code` is treated as the absolute source of truth for product identity, so two different codes sharing a description (e.g. `RETRO PLASTIC 70'S TRAY`) are two genuinely different products that happen to be labelled alike — not a data error. Correcting it would mean merging distinct products. Note this is the mirror image of issue 3, where the code is the identity and the *description* is the unreliable field, which is why 3 is corrected and 4 is not. Left as-is, 27 descriptions still map to more than one stock code, and that is the intended outcome.

**7 — Outliers.** Kept deliberately. The downstream model is tree-based (XGBoost), which splits on feature thresholds rather than fitting distances or coefficients, so it is inherently robust to extreme values and handles them without help. Clipping, winsorising, or dropping them would discard real purchasing behaviour — a genuinely large first order is signal for churn, not noise. The Tukey 1.5×IQR rule flags ~4–8% of rows per column (`quantity` 4.16%, `price` 8.35%, `line_total` 6.15%), all of which stay in.

### A note on issue 8

Worth knowing why `TEST001` needs its own correction rather than falling out of issue 5: the code is letters+digits, so §5's format check classifies it as a valid product variant alongside real codes like `84031A`. Three of its four rows are removed incidentally by other corrections (two have `price == 0` → issue 2; one is a `C` cancellation → issue 6), so it would be easy to assume the problem is handled. The fourth survives everything, and each of the 4 rows is a customer's *entire* first transaction — so without `remove_test_products`, one wholly fabricated customer (12346) reaches the modelling data labelled `churn = 0`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the src/ modules importable from the notebooks/ directory.
SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from correct_data_issues import IN_FILE, correct_data_issues
from data_validation_tests import validate_all, CHECKS

## 1. Run the transformations

Load the first-transaction table (`data/processed/first_transaction_churn.csv`) and apply `correct_data_issues`. The before/after shapes confirm the corrections actually ran.

In [2]:
raw = pd.read_csv(IN_FILE, parse_dates=['invoice_date'])
clean = correct_data_issues(raw)

print(f'Before corrections : {raw.shape}   ({raw["customer_id"].nunique():,} customers)')
print(f'After corrections  : {clean.shape}   ({clean["customer_id"].nunique():,} customers)')
print(f'Rows removed       : {len(raw) - len(clean):,}')

Before corrections : (126080, 10)   (5,346 customers)
After corrections  : (125042, 10)   (5,044 customers)
Rows removed       : 1,038


## 2. Validate the corrections

Run every validation check on the corrected data. `validate_all` raises a `DataValidationError` (listing all failing checks) if any issue remains; if it returns without raising, all corrections are confirmed completed.

In [3]:
validate_all(clean)
print(f'All {len(CHECKS)} validation checks passed on {len(clean):,} rows '
      f'({clean["customer_id"].nunique():,} customers) — no known issues remain.')

All 5 validation checks passed on 125,042 rows (5,044 customers) — no known issues remain.
